In [0]:
import re
import unicodedata
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

RAW_PATH   = "/Volumes/immo/landing/dvf_raw/"
CHECKPOINT = "/Volumes/immo/bronze/checkpoints/dvf_bronze"
TARGET     = "immo.bronze.dvf_mutations"

# Volume pour stocker l'état d'Auto Loader (fichiers déjà traités)
spark.sql("CREATE VOLUME IF NOT EXISTS immo.bronze.checkpoints")

# --- 1. Normalisation des noms de colonnes -------------------------------
def to_snake(name: str) -> str:
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
    return f"c_{s}" if s[0].isdigit() else s

first_file = [f.path for f in dbutils.fs.ls(RAW_PATH) if f.name.endswith(".txt")][0]
raw_header = dbutils.fs.head(first_file, 4000).split("\n")[0].strip()
columns = [to_snake(c) for c in raw_header.split("|")]

assert len(columns) == len(set(columns)), "Doublon dans les noms de colonnes !"
schema = StructType([StructField(c, StringType(), True) for c in columns])

# --- 2. Lecture incrémentale avec Auto Loader ----------------------------
bronze_df = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", "|")
        .option("pathGlobFilter", "*.txt")
        .schema(schema)
        .load(RAW_PATH)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
)

# --- 3. Écriture en Delta -------------------------------------------------
(
    bronze_df.writeStream
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(TARGET)
        .awaitTermination()
)

print(f"{spark.table(TARGET).count():,} lignes dans {TARGET}")